In [ ]:

import ast
import math
import operator
from pathlib import Path

from dotenv import load_dotenv
from ddgs import DDGS
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI


# --------------------------------------------------
# 1. LOAD ENVIRONMENT VARIABLES
# --------------------------------------------------

load_dotenv()

SAMPLE_DIR = (Path(__file__).parent / "sample_files").resolve()


# --------------------------------------------------
# 2. TOOL 1: DUCKDUCKGO WEB SEARCH
# --------------------------------------------------

@tool
def web_search(query: str) -> str:
    """Search the internet using DuckDuckGo and return relevant results."""

    try:
        results = list(DDGS().text(query, max_results=5))

        if not results:
            return "No search results found."

        output = []

        for index, result in enumerate(results, start=1):
            output.append(
                f"{index}. {result.get('title', 'No title')}\n"
                f"URL: {result.get('href', 'No URL')}\n"
                f"Summary: {result.get('body', 'No summary')}"
            )

        return "\n\n".join(output)

    except Exception as error:
        return f"Web search error: {error}"


# --------------------------------------------------
# 3. TOOL 2: PYTHON CALCULATOR
# --------------------------------------------------

@tool
def calculator(expression: str) -> str:
    """Calculate mathematical expressions using safe arithmetic operations."""

    operations = {
        ast.Add: operator.add,
        ast.Sub: operator.sub,
        ast.Mult: operator.mul,
        ast.Div: operator.truediv,
        ast.FloorDiv: operator.floordiv,
        ast.Mod: operator.mod,
        ast.Pow: operator.pow,
    }

    unary_operations = {
        ast.UAdd: operator.pos,
        ast.USub: operator.neg,
    }

    def evaluate(node):

        if isinstance(node, ast.Expression):
            return evaluate(node.body)

        elif isinstance(node, ast.Constant):
            if type(node.value) not in (int, float):
                raise ValueError("Only numeric values are allowed.")
            return node.value

        elif isinstance(node, ast.BinOp):
            operation = operations.get(type(node.op))

            if operation is None:
                raise ValueError("Unsupported mathematical operation.")

            left = evaluate(node.left)
            right = evaluate(node.right)

            if isinstance(node.op, ast.Pow) and abs(right) > 10:
                raise ValueError("Exponent is too large.")

            return operation(left, right)

        elif isinstance(node, ast.UnaryOp):
            operation = unary_operations.get(type(node.op))

            if operation is None:
                raise ValueError("Unsupported unary operation.")

            return operation(evaluate(node.operand))

        else:
            raise ValueError("Invalid mathematical expression.")

    try:
        tree = ast.parse(expression, mode="eval")
        result = evaluate(tree)

        if not math.isfinite(result) or abs(result) > 1e100:
            return "Error: Result is too large or invalid."

        return f"Calculation result: {result}"

    except Exception as error:
        return f"Calculator error: {error}"


# --------------------------------------------------
# 4. TOOL 3: LOCAL FILE READER
# --------------------------------------------------

@tool
def read_local_file(filename: str) -> str:
    """Read a text, Markdown or CSV file from the sample_files folder."""

    try:
        if Path(filename).name != filename:
            return "Error: Only filenames are allowed."

        if Path(filename).suffix.lower() not in (".txt", ".md", ".csv"):
            return "Error: Only TXT, MD and CSV files are supported."

        file_path = (SAMPLE_DIR / filename).resolve()

        if not file_path.is_relative_to(SAMPLE_DIR):
            return "Error: Access outside the sample_files folder is forbidden."

        if not file_path.is_file():
            return f"File not found: {filename}"

        if file_path.stat().st_size > 100_000:
            return "Error: File exceeds the 100 KB size limit."

        content = file_path.read_text(encoding="utf-8")

        return f"Contents of {filename}:\n\n{content}"

    except Exception as error:
        return f"File reader error: {error}"


# --------------------------------------------------
# 5. INITIALIZE GEMINI MODEL
# --------------------------------------------------

model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0
)



# --------------------------------------------------
# 6. REGISTER ALL TOOLS
# --------------------------------------------------

tools = [
    web_search,
    calculator,
    read_local_file
]


# --------------------------------------------------
# 7. CREATE AUTONOMOUS AI AGENT
# --------------------------------------------------

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt="""
    You are an intelligent autonomous AI assistant.

    You have access to three tools:

    1. web_search:
       Use this tool to find current information on the internet.

    2. calculator:
       Use this tool for mathematical calculations.

    3. read_local_file:
       Use this tool to read files stored in the sample_files folder.

    Instructions:
    - Select the appropriate tool based on the user's request.
    - Use tools whenever the task requires them.
    - You may use multiple tools for a single question.
    - Use the tool results to prepare your final answer.
    - Never invent tool results.
    - If a tool fails, clearly explain the problem.
    - Give clear and helpful final answers.
    """
)


# --------------------------------------------------
# 8. RUN THE AGENT
# --------------------------------------------------

def run_agent(user_input):

    print("\n" + "=" * 60)
    print("AUTONOMOUS AI AGENT")
    print("=" * 60)

    try:
        response = agent.invoke(
            {
                "messages": [
                    {"role": "user", "content": user_input}
                ]
            }
        )

        # Display tool actions and observations
        for message in response["messages"]:

            tool_calls = getattr(message, "tool_calls", [])

            for call in tool_calls:
                print(f"\n[ACTION] Tool: {call['name']}")
                print(f"Arguments: {call['args']}")

            if getattr(message, "type", "") == "tool":
                print(f"\n[OBSERVATION]\n{message.content}")

        # Display final response
        print("\n" + "-" * 60)
        print("FINAL ANSWER")
        print("-" * 60)

        print(response["messages"][-1].content)

    except Exception as error:
        print(f"\nAgent execution error: {error}")


# --------------------------------------------------
# 9. INTERACTIVE CONSOLE
# --------------------------------------------------

if __name__ == "__main__":

    print("\nWelcome to the Multi-Tool Autonomous AI Agent!")
    print("Available tools: Web Search, Calculator, File Reader")
    print("Type 'exit' to close the application.")

    while True:

        question = input("\nYou: ").strip()

        if question.lower() in ("exit", "quit"):
            print("Agent stopped. Goodbye!")
            break

        if not question:
            print("Please enter a question.")
            continue

        run_agent(question)


# Task 3: Multi-Tool Autonomous AI Agent

## Project Overview

This project demonstrates an autonomous AI agent built using Python, LangChain, and Google Gemini.

The agent can independently select and use different tools based on the user's query.

## Tools Used

1. **DuckDuckGo Web Search** – Searches the internet for information.
2. **Python Calculator** – Performs mathematical calculations.
3. **Local File Reader** – Reads and summarizes local text files.

## Technologies Used

- Python 3.11
- LangChain
- Google Gemini API
- DuckDuckGo Search
- python-dotenv

## 1. Import Required Libraries

The following libraries are used to build the autonomous AI agent.

In [2]:
import ast
import math
import operator
from pathlib import Path
import os

from dotenv import load_dotenv
from ddgs import DDGS
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

## 2. Load Environment Variables

Load the Google Gemini API key securely from the `.env` file.

In [3]:
# Load environment variables
load_dotenv()

# Retrieve Google Gemini API key
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# Verify API key availability without displaying it
if GOOGLE_API_KEY:
    print("Google Gemini API key loaded successfully!")
else:
    print("API key not found. Please check your .env file.")

Google Gemini API key loaded successfully!


## 3. Initialize Google Gemini Model

Configure the Gemini model that will power the autonomous AI agent.

In [4]:
# Initialize Google Gemini model
model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0
)

print("Gemini model initialized successfully!")

Gemini model initialized successfully!


## 4. Define the Python Calculator Tool

The calculator safely evaluates mathematical expressions using Python's AST module.

In [5]:
def calculator(expression):
    """Safely evaluate mathematical expressions."""

    allowed_operators = {
        ast.Add: operator.add,
        ast.Sub: operator.sub,
        ast.Mult: operator.mul,
        ast.Div: operator.truediv,
        ast.Pow: operator.pow,
        ast.Mod: operator.mod,
        ast.FloorDiv: operator.floordiv,
        ast.USub: operator.neg,
        ast.UAdd: operator.pos,
    }

    def evaluate(node):
        if isinstance(node, ast.Constant) and type(node.value) in (int, float):
            return node.value

        elif isinstance(node, ast.BinOp) and type(node.op) in allowed_operators:
            return allowed_operators[type(node.op)](
                evaluate(node.left),
                evaluate(node.right)
            )

        elif isinstance(node, ast.UnaryOp) and type(node.op) in allowed_operators:
            return allowed_operators[type(node.op)](evaluate(node.operand))

        else:
            raise ValueError("Unsupported mathematical expression")

    try:
        tree = ast.parse(expression, mode="eval")
        result = evaluate(tree.body)
        return str(result)

    except Exception as e:
        return f"Calculation error: {e}"

In [6]:
calculator("1250 * 48 + 750")

'60750'

## 5. Define the DuckDuckGo Web Search Tool

This tool searches the web and returns the top five results for a given query.

In [7]:
def web_search(query):
    """Search the web using DuckDuckGo."""

    try:
        results = DDGS().text(query, max_results=5)

        if not results:
            return "No search results found."

        formatted_results = []

        for result in results:
            formatted_results.append(
                f"Title: {result.get('title', '')}\n"
                f"URL: {result.get('href', '')}\n"
                f"Summary: {result.get('body', '')}"
            )

        return "\n\n".join(formatted_results)

    except Exception as e:
        return f"Web search error: {e}"

## 6. Define the Local File Reader Tool

This tool reads local text, Markdown, and CSV files from the sample_files directory.

In [8]:
def read_local_file(filename):
    """Read a local file from the sample_files directory."""

    base_dir = Path("sample_files").resolve()
    file_path = (base_dir / filename).resolve()

    try:
        # Prevent access outside the sample_files directory
        if not file_path.is_relative_to(base_dir):
            return "Error: Access outside sample_files is not allowed."

        # Allow only supported file types
        if file_path.suffix.lower() not in [".txt", ".md", ".csv"]:
            return "Error: Only TXT, MD, and CSV files are allowed."

        if not file_path.is_file():
            return "Error: File not found."

        # Limit file size to 100 KB
        if file_path.stat().st_size > 100 * 1024:
            return "Error: File exceeds the 100 KB size limit."

        return file_path.read_text(encoding="utf-8")

    except Exception as e:
        return f"File reading error: {e}"

In [9]:
print(read_local_file("sample.txt"))

Python Generative AI Internship

Task 3: Multi-Tool Autonomous AI Agent

This project uses Google Gemini and LangChain
to build an autonomous AI assistant.

The agent has three tools:

1. DuckDuckGo Web Search
2. Python Calculator
3. Local File Reader

The agent selects the appropriate tool
depending on the user's question.


## 7. Register Tools with the AI Agent

Register the calculator, web search, and local file reader functions as tools for the autonomous agent.

In [10]:
from langchain.tools import tool

@tool
def calculator_tool(expression: str) -> str:
    """Perform mathematical calculations. Input should be a mathematical expression."""
    return calculator(expression)


@tool
def web_search_tool(query: str) -> str:
    """Search the internet for current information about a topic."""
    return web_search(query)


@tool
def file_reader_tool(filename: str) -> str:
    """Read a local TXT, MD, or CSV file from the sample_files folder."""
    return read_local_file(filename)


tools = [
    calculator_tool,
    web_search_tool,
    file_reader_tool
]

print("All three tools registered successfully!")

All three tools registered successfully!


## 8. Create the Autonomous AI Agent

The LangChain agent uses Google Gemini to understand user queries, select suitable tools, and combine their results into a final response.

In [11]:
# Create the autonomous AI agent
agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=(
        "You are a helpful autonomous AI assistant. "
        "Use the available tools whenever needed. "
        "Use the calculator for mathematical questions, "
        "web search for current or external information, "
        "and file reader for questions about local files. "
        "You may use multiple tools to solve complex tasks. "
        "Provide clear and accurate final answers."
    )
)

print("Autonomous AI Agent created successfully!")

Autonomous AI Agent created successfully!


## 9. Test the Autonomous AI Agent

### Test 1: Mathematical Calculation

Check whether the agent automatically selects the Python Calculator tool.

In [12]:
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "Calculate 1250 * 48 + 750"}
    ]
})

print(response["messages"][-1].content)

[{'type': 'text', 'text': '1250 * 48 + 750 = **60,750**', 'extras': {'signature': 'EqUCCqICAWkUfRNJd/xKe2MYaVPlRpIh4fi4EIGU/CeLMYJuMV6cFDpWfr8eP6PN3ILNi+fSJzzVTY5PuCQVBGoUezDi3RwNULm6VFXBXXcOadKVGOE4azhatcvqJxdJMGb2MGWQZNwGDSIAl8vw5JlODj4g8FSraMmEb2knxtc2Wsu9HdldqTJ1764hyepmSqr+rwge2DO3/70AxoBbyasLcwF2nuLk3oz4u6g2MF19S8bzA3mWuvlMVeJtulP8JLdfcVYn9XzrPPPSRt9WurSW8foTtT6aYSSsifavKyhbOTqlcQW1RvBZGBpmY6ltKceAxB+1uV0VbKPTbZ7hHX7s0ovKZc6kpOrJpBWNz7OwsEPeDCdKCMtm6oSia4k+W5oowp9J7nU='}}]


### Test 2: Local File Reading

Check whether the agent automatically selects the Local File Reader tool and summarizes the file.

In [13]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Read sample.txt and summarize its contents."
        }
    ]
})

print(response["messages"][-1].content)

[{'type': 'text', 'text': "Here is a summary of the contents of **sample.txt**:\n\nThe file describes **Task 3** of a **Python Generative AI Internship**, focused on building a **Multi-Tool Autonomous AI Agent** using Google Gemini and LangChain. \n\nThe AI agent is equipped with three tools:\n1. **DuckDuckGo Web Search**\n2. **Python Calculator**\n3. **Local File Reader**\n\nThe agent autonomously selects and uses the appropriate tool based on the user's prompt or question.", 'extras': {'signature': 'EoQECoEEAWkUfRMbkJDp9oFrg+/Z8CUFzOz6IN/l095pPEbyaibPze2k2QBas2JnUEqzP5xhp80ohm8mOfvd0VjBuFO6QAgEzzu/Y8hceh6SHFpMh+59mT2VXOi39yN98/Nz2ZRuUagVqk8928+SjC9igbz92IkK3BIFTjl+rCZQT3tNBqoCakTB3OQUqjoKlbpSETgDEisssMlG7FHiifo1+7wkquMfzqQ5yr8BG5nEw5STrO2ffgPVehgBZMjqVlbQMk0ebdfWIWoCZh3DjLwLQMMWg6fh1sfU172CTBH7Fj5pTuUR0/6dXRYfvpYqaj4cbx6BBo1lc9r4LFvQmtHwspnmWCv3TlRBbu6FT3MTpUQRzYVtgyHivf+oV92rlJglr06I9j/gx7PER68flpC0m3nNqEZcQxYr7AqcPhMoTVSVMUO3NMnDin8txEBKu7Z2HKPQP21LxCXJovit6Dtsm2M/Z4460z8QNSl9lIAg6

### Test 3: Web Search

Check whether the agent automatically selects DuckDuckGo Web Search to retrieve external information.

In [14]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Search the web for the latest developments in Generative AI in 2026 and give me 5 key points."
        }
    ]
})

print(response["messages"][-1].content)

[{'type': 'text', 'text': 'Based on the latest industry insights, research reports, and enterprise developments in 2026, here are **5 key points** highlighting the major advancements in Generative AI:\n\n---\n\n### **1. Shift to Autonomous Agentic AI & Multiagent Workflows**\n* **From Chatbots to Autonomous Agents:** Generative AI has transitioned from conversational Q&A tools to autonomous agents capable of multi-step planning, tool usage, and task execution with minimal human supervision.\n* **Orchestrated Systems:** Enterprise applications increasingly deploy multiagent architectures where specialized AI agents collaborate across workflows—managing everything from code generation and software testing to customer support and supply chain logistics.\n\n---\n\n### **2. Native Multimodality & Physical AI Integration**\n* **Unified Multimodal Architectures:** Foundation models in 2026 natively process and generate text, high-definition video, spatial 3D environments, audio, and sensor da

### Test 4: Multi-Tool Problem Solving

Verify whether the agent can read a local file, identify its tools, perform a calculation, and combine the results into one report.

In [15]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": (
                "Read sample.txt, identify the three tools mentioned in it, "
                "calculate 2500 * 36 + 1500, and provide a combined report "
                "with the tool names and calculation result."
            )
        }
    ]
})

print(response["messages"][-1].content)

[{'type': 'text', 'text': 'Here is the report based on the file content and calculation:\n\n---\n\n### **Summary Report**\n\n#### **1. Tools Identified in `sample.txt`**\n1. **DuckDuckGo Web Search**\n2. **Python Calculator**\n3. **Local File Reader**\n\n---\n\n#### **2. Mathematical Calculation Result**\n* **Expression:** $2500 \\times 36 + 1500$\n* **Calculation:** $90,000 + 1,500$\n* **Result:** **91,500**', 'extras': {'signature': 'ErQECrEEAWkUfRNtZyxEKKKn+w8FwHtLbWnBJdlwM+3C8KLCeVviemolbby8bUvFkMVyqYfMpgTVL80EK5EuFyuTq1Jg+SSe8yF1FX89kZX/mYOQfvNuiWFhlAylNOl8o+6/DCRzc1rNtVvkKHQunjCvxJopVPl5P13sU3XrRNlBtnnKwQ8NS39xS2ptCfhFaL+da99CnkawxO+koXKYUA7+zH7+JQRcEkDqN2254Oe/bQe5K3NB0jAzLgNca4QdiBSMSpCusDyUnbOpOjIziJpBeKRG7jzVNGr9+c6ZGTaTh1406zz9t+W4gCK8kJ5/6xYeM+3I7ntgfqTbIa60kHCMjQyFBYVQnpRP5tqPkoKBH47Xf7GoQVrlzTN8oBHYAiO8gyMInt40aTxMGAagQ+k13hnW3uobVSjJYv9CXZuEB7TQzb/3vkZCjYmpMw2cx8ICIOv9Qxe+yyNIjSjzWhPU2uY7gpD7be3uF9uUPdlPn1g3tLD8WEg+lPIdyLVobl/DanUCwaHtiDVf1hNFb+kKVJB6XzMHnpvFjZPnNql3ZSFH

## 10. Conclusion

The Multi-Tool Autonomous AI Agent was successfully developed using Python, LangChain, and Google Gemini.

The agent can independently select and use three tools:

- DuckDuckGo Web Search
- Python Calculator
- Local File Reader

All four functional tests were completed successfully, demonstrating the agent's ability to perform calculations, retrieve web information, read local files, and solve multi-step tasks using multiple tools.

This project demonstrates the practical implementation of autonomous AI agents and tool-based problem solving.